# 03 - nn.Module: Building Neural Networks

**Goal:** Learn PyTorch's way of organizing neural networks.

---

## Why nn.Module?

In notebook 02, we manually created parameters:
```python
w = torch.tensor([0.5], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
```

For real networks with millions of parameters, this doesn't scale. `nn.Module` provides:
- Automatic parameter management
- Easy GPU transfer
- Save/load functionality
- Layer composition

## Built-in Layers

PyTorch provides ready-to-use layers in `torch.nn`:

In [ ]:
import torch
import torch.nn as nn

# Linear layer (fully connected)
# y = x @ W.T + b
linear = nn.Linear(in_features=10, out_features=5)

print(f"Linear layer: {linear}")
print(f"Weight shape: {linear.weight.shape}")  # (out, in) = (5, 10)
print(f"Bias shape: {linear.bias.shape}")      # (out,) = (5,)

# Use it
x = torch.randn(3, 10)  # Batch of 3, each with 10 features
y = linear(x)
print(f"\nInput: {x.shape} → Output: {y.shape}")

In [ ]:
# Common layers you'll use:

# Convolutional layers
conv2d = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1)
conv3d = nn.Conv3d(in_channels=1, out_channels=24, kernel_size=3, padding=1)  # For 3D volumes

# Pooling
maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

# Normalization
batchnorm = nn.BatchNorm2d(num_features=64)

# Activation functions
relu = nn.ReLU()
sigmoid = nn.Sigmoid()
softmax = nn.Softmax(dim=1)

# Dropout (regularization)
dropout = nn.Dropout(p=0.5)

print("These are all nn.Module subclasses")
print(f"conv2d parameters: {sum(p.numel() for p in conv2d.parameters()):,}")

## Creating Custom Models

Inherit from `nn.Module` to create your own networks:

```python
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Define layers here
    
    def forward(self, x):
        # Define forward pass here
        return x
```

In [ ]:
# Simple feedforward network

class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()  # Always call this first!
        
        # Define layers as attributes
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.layer2 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        # Define how data flows through the network
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

# Create model
model = SimpleNet(input_size=784, hidden_size=128, output_size=10)
print(model)

In [ ]:
# Test the model
x = torch.randn(32, 784)  # Batch of 32, each with 784 features
y = model(x)              # Calls forward() automatically

print(f"Input: {x.shape}")
print(f"Output: {y.shape}")

## Accessing Parameters

Every nn.Module keeps track of all its learnable parameters:

In [ ]:
# List all parameters
print("Parameters in the model:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.shape}")

# Total count
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
# Check requires_grad (automatically True for nn.Module parameters)
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

## Moving to GPU

One call moves all parameters to GPU:

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Move model to device
model = model.to(device)

# Check where parameters are
print(f"layer1.weight device: {model.layer1.weight.device}")

# Input must also be on same device!
x = torch.randn(32, 784, device=device)
y = model(x)
print(f"Output device: {y.device}")

## Training vs Evaluation Mode

Some layers behave differently during training vs inference:
- **Dropout**: drops neurons during training, does nothing during inference
- **BatchNorm**: uses batch statistics during training, running statistics during inference

Switch modes with `.train()` and `.eval()`:

In [ ]:
class ModelWithDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 10)
        self.dropout = nn.Dropout(p=0.5)  # 50% dropout
    
    def forward(self, x):
        x = self.linear(x)
        x = self.dropout(x)
        return x

model = ModelWithDropout()
x = torch.ones(1, 10)

# Training mode (default)
model.train()
print(f"Training mode: {model.training}")
y_train = model(x)
print(f"Output (some zeros from dropout): {y_train}")

# Evaluation mode
model.eval()
print(f"\nEval mode: {model.training}")
y_eval = model(x)
print(f"Output (no dropout): {y_eval}")

## Saving and Loading Models

Save the learned parameters (what's in the `.pth` files):

In [ ]:
# Create and "train" a model (we'll just modify weights manually)
model = SimpleNet(784, 128, 10)
model.layer1.weight.data.fill_(0.5)  # Set all weights to 0.5

# Save
torch.save(model.state_dict(), '/tmp/model.pth')
print("Model saved!")
print(f"state_dict keys: {model.state_dict().keys()}")

In [ ]:
# Load into a new model
new_model = SimpleNet(784, 128, 10)  # Same architecture!
print(f"Before loading - layer1.weight[0,0]: {new_model.layer1.weight[0,0].item():.4f}")

new_model.load_state_dict(torch.load('/tmp/model.pth'))
print(f"After loading - layer1.weight[0,0]: {new_model.layer1.weight[0,0].item():.4f}")

## nn.Sequential: Quick Model Building

For simple sequential architectures:

In [ ]:
# Instead of defining a class:
model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

# Use it
x = torch.randn(32, 784)
y = model(x)
print(f"\nInput: {x.shape} → Output: {y.shape}")

## A CNN Example

Putting together what we learned in Phase 0:

In [ ]:
class SimpleCNN(nn.Module):
    """CNN for 28x28 grayscale images (like MNIST)"""
    
    def __init__(self, num_classes=10):
        super().__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)   # 28x28 -> 28x28
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 14x14 -> 14x14
        
        self.pool = nn.MaxPool2d(2, 2)  # Halves spatial dimensions
        self.relu = nn.ReLU()
        
        # After conv1+pool: 28x28 -> 14x14
        # After conv2+pool: 14x14 -> 7x7
        # Flatten: 64 * 7 * 7 = 3136
        self.fc = nn.Linear(64 * 7 * 7, num_classes)
    
    def forward(self, x):
        # x shape: (batch, 1, 28, 28)
        
        x = self.pool(self.relu(self.conv1(x)))  # -> (batch, 32, 14, 14)
        x = self.pool(self.relu(self.conv2(x)))  # -> (batch, 64, 7, 7)
        
        x = x.view(x.size(0), -1)  # Flatten: -> (batch, 3136)
        x = self.fc(x)             # -> (batch, 10)
        
        return x

model = SimpleCNN()
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Test with a batch of images
batch = torch.randn(16, 1, 28, 28)  # 16 grayscale 28x28 images
output = model(batch)

print(f"Input: {batch.shape}")
print(f"Output: {output.shape}")
print(f"Output for first image: {output[0]}")
print(f"Predicted class: {output[0].argmax().item()}")

## Connection to Production Code

Your production code (`segmentation/src/inference_engine_v016.py`) does exactly this:

```python
from monai.networks.nets import SwinUNETR

# Create model architecture
model = SwinUNETR(
    img_size=[96, 96, 96],
    in_channels=1,
    out_channels=nb_classes,
    feature_size=24,
)

# Load trained weights
model.load_state_dict(torch.load('best_metric_model.pth'))

# Move to GPU
model = model.to(device)

# Set to evaluation mode
model.eval()

# Inference
with torch.no_grad():
    prediction = model(input_volume)
```

Same pattern!

## Summary

| Concept | What it does |
|---------|-------------|
| `nn.Module` | Base class for all neural networks |
| `__init__` | Define layers as attributes |
| `forward()` | Define how data flows through |
| `.parameters()` | Get all learnable parameters |
| `.to(device)` | Move model to GPU/CPU |
| `.train()` / `.eval()` | Switch between modes |
| `state_dict()` | Dict of all parameters (for save/load) |
| `nn.Sequential` | Quick way to stack layers |

**Next:** The complete training loop with optimizers and loss functions.